[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/12_linear_attention.ipynb)

# 🔴 困难：线性自注意力（Linear Self-Attention）

实现 **线性注意力（Linear Attention）** —— 时间复杂度从 O(S²·D) 降低到 O(S·D²)，支持高效的长序列处理。

将 softmax 替换为 **核特征映射** $\phi$：

$$\text{LinearAttn}(Q,K,V) = \frac{\phi(Q) \left(\phi(K)^T V\right)}{\phi(Q) \cdot \sum \phi(K)}$$

### 特征映射
使用 $\phi(x) = \text{elu}(x) + 1$（确保特征非负）。

### 函数签名
```python
def linear_attention(Q, K, V):
    # Q: (B, S, D_k), K: (B, S, D_k), V: (B, S, D_v)
    # 返回值: (B, S, D_v)
```

### 核心思想
不计算 S×S 的注意力矩阵，而是先计算 $\phi(K)^T V$（得到一个 D_k×D_v 的矩阵），再与 $\phi(Q)$ 相乘。

### 要求
- **必须** 使用特征映射（**禁止** 使用 softmax）
- 必须实现 O(S·D²) 复杂度 —— 在长序列上应运行迅速
- **允许** 使用 `F.elu`

In [ ]:
# 在 Colab 中安装 torch-judge（在 JupyterLab/Docker 中无操作）
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import torch.nn.functional as F

F.relu(input):
- 计算开销极低：\
不需要像 Sigmoid 或 Tanh 那样进行复杂的指数运算（$e^x$），只需一个阈值判断。
- 缓解梯度消失：\
在 $x > 0$ 的区域，导数恒为 1。这意味着梯度可以无损地传递到前面的层，不会像 Sigmoid 那样在深层网络中迅速衰减。
- 稀疏激活：\
它会让一部分神经元的输出变为 0，从而使网络具有稀疏性（Sparsity），有助于提取关键特征并减少参数依赖。
$$\text{ReLU}(x) = \max(0, x)$$

$$\text{ReLU}'(x) = 
\begin{cases} 
1 & \text{if } x > 0 \\
0 & \text{if } x < 0 
\end{cases}$$

在 $x=0$ 处，ReLU 理论上不可导，但在实际工程实现中（如 PyTorch），通常人为定义其在 0 点的导数为 0 或 0.5，这并不影响神经网络的训练。

F.leaky_relu(input, negative_slope):
- 默认值：在 PyTorch 的 F.leaky_relu 中，默认值是 0.01。
- 常见范围：通常设置在 $0.01$ 到 $0.2$ 之间。
- 特殊变体：如果这个参数不是人工设定的，而是让网络自己通过学习得到的，那么这种激活函数被称为 PReLU (Parametric ReLU)。
$$\text{Leaky ReLU}(x) = \max(negative\_slope * x, x)$$

F.elu(input, alpha): Exponential Linear Unit, 指数线性单元
核心特性与功能
- 均值接近于零：\
与 ReLU 不同，ELU 在负值区域有输出。这使得激活值的平均值更接近于零，从而减少了“偏置偏移”（Bias Shift）效应，有助于加快梯度的收敛速度。
- 解决 Dead ReLU 问题：\
ReLU 在 $x < 0$ 时梯度为 0，可能导致神经元“死亡”。ELU 在负值区域具有非零梯度，允许神经元在训练过程中持续更新。
- 左侧软饱和性： \
当 $x \to -\infty$ 时，ELU 会收敛至 $-\alpha$。这种饱和特性使得模型对输入中的噪声具有更强的鲁棒性（Robustness）。
- 平滑性： \
在 $x=0$ 处，如果 $\alpha=1$，ELU 是连续且可导的（$C^1$ 连续），这比 ReLU 在 0 点的硬转折更利于优化。
$$\text{ELU}(x) = 
\begin{cases} 
x & \text{if } x > 0 \\
\alpha (e^x - 1) & \text{if } x \le 0 
\end{cases}$$

F.gelu(input, approximate)：（Gaussian Error Linear Unit，高斯误差线性单元）approximate = 'none' 表示使用最高原公式(包含 erf)，'tanh'表示使用 tanh  \
相比于传统的 ReLU，GELU 的主要特点如下：
- 非零梯度：在 $x < 0$ 的区域，GELU 仍然保留了微小的梯度（不同于 ReLU 直接截断为 0），这有助于权重的更新。
- 平滑性：GELU 在全域范围内是连续可微的，这使得优化过程更加平滑。
- 概率特性：它本质上是根据输入值的大小，随机决定是否将输入置零。输入越小，被置零的概率越大；输入越大，越有可能被保留。

GELU 的精确数学表达式为：$$GELU(x) = x \cdot P(X \le x) = x \Phi(x) = x \cdot \frac{1}{2} \left[ 1 + \text{erf} \left( \frac{x}{\sqrt{2}} \right) \right]$$其中：$\Phi(x)$ 是标准正态分布的累积分布函数（CDF）。$\text{erf}$ 是误差函数。\
工业界使用高精度近似公式:
- 近似公式 A（最常用）：基于 $\tanh$ 函数的近似，计算速度快且足够精确：$$GELU(x) \approx 0.5x \left( 1 + \tanh \left[ \sqrt{\frac{2}{\pi}} (x + 0.044715x^3) \right] \right)$$
- 近似公式 B：基于 $\sigma$ (Sigmoid) 函数的近似：$$GELU(x) \approx x \cdot \sigma(1.702x)$$

线性注意力机制是将指数线性单元 elu (Exponential Linear Unit) 替换 softmax 以实现, \
时间复杂度为 O(2 * b * seq_len * dim^2 + 2 * b * seq_len * dim) 相当于 O(2 * b * s * d^2)
$$\text{LinearAttn}(Q,K,V) = \frac{\phi(Q) \left(\phi(K)^T V\right)}{\phi(Q) \cdot \sum \phi(K)}$$

线性注意力通过特征映射 $\phi$ 移除指数操作 \
在长序列场景下（例如 $S=10000, D=64$），计算效率的提升是巨大的：\
标准 Attention: $S^2 \times D \approx 10^8 \times 64$ 次操作。 \
线性 Attention: $S \times D^2 \approx 10^4 \times 4096$ 次操作。

由于没有了中间的 $S \times S$ 矩阵，我们实际上是在计算：$$\text{Output} = \frac{Q(K^T V)}{\text{Normalization}}$$

In [ ]:
# ✏️ 在此处实现你的代码

def linear_attention(Q, K, V):
    """
    线性自注意力实现
    复杂度: O(S * D_k * D_v)
    
    参数:
        Q: (B, S, D_k)
        K: (B, S, D_k)
        V: (B, S, D_v)
    """
    # 1. 对特征应用特征映射 phi(x) = elu(x) + 1, 确保特征为非负数，这是替代 Softmax 概率分布的关键
    phi_Q = F.elu(Q) + 1
    phi_K = F.elu(K) + 1

    # 2. 计算上下文矩阵 (Context Matrix)
    # 核心优化：先计算 (phi_K.T @ V)，维度为 (B, D_k, D_v)
    # 使用 einsum 可以清晰地处理 Batch 维度
    # 'bsd, bsv -> bdv' 对应 (B, S, D_k) 和 (B, S, D_v) 相乘
    # 时间复杂度: O(b * dim^2 * seq_len)
    kv_matrix = torch.einsum('bsd, bsv -> bdv', phi_K, V)
    # kv_matrix = torch.bmm(phi_K.transpose(1, 2), V)

    # 3. 计算归一化因子 (Normalizer)
    # 对应公式中的分母：phi(Q) * sum(phi(K))
    # sum(phi_K) 维度为 (B, D_k)
    # 时间复杂度: O(b * seq_len * dim)
    k_sum = phi_K.sum(dim=1) 
    
    # phi(Q) @ k_sum 得到每个 query 的归一化系数，维度为 (B, S, 1)
    # 'bsd, bd -> bs'
    # 时间复杂度: O(b * seq_len * dim)
    denominators = torch.einsum('bsd, bd -> bs', phi_Q, k_sum).unsqueeze(-1)
    # denominator = torch.bmm(phi_Q, k_sum)

    # 4. 计算最终输出
    # 分子：phi(Q) @ (phi_K.T @ V)，维度为 (B, S, D_v)
    # 'bsd, bdv -> bsv'
    # 时间复杂度: O(b * seq_len * dim^2), v = dim
    numenators = torch.einsum('bsd, bdv -> bsv', phi_Q, kv_matrix)
    # numenator = torch.bmm(phi_Q, kv_matrix)

    # 5. 归一化并返回
    return numenators / (denominators + 1e-6) # 添加微小值防止除以零

In [ ]:
# 🧪 调试测试
Q = torch.randn(1, 8, 16)
K = torch.randn(1, 8, 16)
V = torch.randn(1, 8, 32)
out = linear_attention(Q, K, V)
print("输出形状：", out.shape)   # 应为 (1, 8, 32)
print("包含 NaN？", torch.isnan(out).any().item())

In [ ]:
from torch_judge import check
check('linear_attention')